# 1. A standard FTH experiment at normal incidence

Follow sample design → local interaction and multislice → exit field → physical detector → acquisition. Figure 1 sketches the experiment and code flow; Figure 2 displays the simulated stages. CR and CL use identical geometry and one shared synthetic photon scale. [API guide](../API_GUIDE.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/scattering_calculator').is_dir())
PAPER = ROOT / 'paper/scattering_calculator'
sys.path.insert(0, str(PAPER))
sys.path.insert(0, str(ROOT / "src"))
from scattering_calculator import simulation_pipelines as sim
import experiments as ex
import workflow as wf

def show(name):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(OUT / (name + '.png')))
    ax.axis('off')
    plt.show()


## Editable configuration
The object opening removes only Au; the reference opening penetrates every layer. Change the sample builder for other masks or magnetization inputs.

## Standard experiment setup

Use the same production configuration sections in every example:
**X-ray source → simulation grid → sample → magnetization → apertures →
illumination → sample propagation → detector/acquisition → beamstop → energy scan**.
Physical lengths are metres, energy is eV, and angles are radians; recipe
thickness literals are nm. See [the setup standard](../../../docs/experiment_setup.md).
Specialized plotting, reconstruction and analytic-reference settings are analysis
controls, separate from the physical experiment definition.


In [ ]:
# 1. X-RAY SOURCE — energy in eV, flux in photons/s, angles in radians
# 2. SIMULATION GRID — (y, x) pixels and physical pitch in metres
# 3. SAMPLE — recipe thickness literals are nm; slice thickness is metres
# 4. MAGNETIZATION — all physical pattern lengths are metres
# 5. APERTURES — positions are (y, x) in metres
# 6. ILLUMINATION — profile, focus and direction
# 7. SAMPLE PROPAGATION — field representation and multislice
# 8. DETECTOR / ACQUISITION — physical geometry, response and artifacts
# 9. BEAMSTOP; 10. ENERGY SCAN; analysis controls remain separate
experiment = sim.ExperimentConfig(
    xray=sim.XRayConfig(energy=778., photon_flux=1., pol="CR",
                       coherence_length=(100e-6, 100e-6)),
    simulation=sim.SimulationConfig(shape=(192,192), real_space_pixel_size=10e-9),
    sample=sim.SampleConfig(recipe="Au(300)/SiN(20)/Pt(2)/Co(10)/Pt(2)",
                            max_slice_thickness=5e-9, sample_name="Normal-incidence FTH"),
    magnetic_pattern=sim.MagneticPatternConfig(
        pattern_type_method="smooth_domains", pattern_config={"period":110e-9}),
    aperture=sim.FrontApertureConfig(aperture_method="FTH_circular", aperture_config={
        "apertures_type":["OH","RH"], "apertures_radius":[180e-9,30e-9],
        "apertures_center":[(0.,0.),(0.,600e-9)], "apertures_sigma":[0.,0.],
        "apertures_top_radius_factor":[1.,1.], "thickness_OH":300e-9,
    }),
    illumination=sim.IlluminationConfig(illumination_function="gaussian", illumination_config={
        "center":(0.,0.), "distance":0., "fwhm":2*np.sqrt(np.log(2))*650e-9,
        "alpha_beam":(0.,0.),
    }),
    propagation=sim.SamplePropagatorConfig(propagator_method="Jones", propagator_config={
        "propagate":True, "dielectric_tensor_compact":True,
    }),
    detector=sim.DetectorConfig(shape=(256,256), pixel_size=13.5e-6,
        sample_to_detector_distance=.05, detector_center=(128,128),
        detector_propagation_method="fraunhofer", use_detector_pixel_footprint=False,
        detector_pixel_footprint_samples=3, analyzer_angle=None,
        measurement_config={"exposure_time":1.,"number_frames":1,"max_counts_per_image":None},
        detector_params={"counts_per_photon":1.,"quantum_efficiency":1.,
            "readout_noise_average":0.,"readout_noise_sigma":2.,"detector_threshold":16000.,
            "noise_seed":17},
        artifacts_config={"sigma_photon":0.,"camera_seed":31,"average_hot_pixels":12.,
                          "average_cold_pixels":12.,"cosmic_rays_per_second":2.}),
    beamstop=sim.BeamstopConfig(bs_method="circular",bs_detector_distance=.01,
                              bs_config={"radius":40.5e-6}),
    energies_eV=tuple(np.arange(772.,801.)),
)
OUT = PAPER / 'results' / 'notebook_fth_basics'


In [ ]:
case, ideal, acquisition = wf.baseline(OUT, experiment, experiment.detector, experiment.detector)
show('fig01_setup_workflow')
show('fig02_fth_chain')


## Inspect the API outputs
Complex exit fields retain phase; intensity alone cannot be propagated coherently. `case["images"]` is on the common FFT q grid. `ideal` and acquisition arrays are on the physical camera. Counts use a synthetic budget, not absolute flux calibration.

In [ ]:
print('Exit fields:', case['exits'].shape)
print('FFT intensities:', case['images'].shape)
print('Physical detector:', ideal.shape)
print('Shared budget scale:', acquisition['scale'])
print('Saved to:', OUT)
